# 📊 Evaluation — Evaluación Completa del Modelo

Evaluación detallada del modelo entrenado: más allá del accuracy.
Incluye matriz de confusión, curvas ROC/PR, análisis de errores.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.preprocessing import label_binarize

from src.modules.iris_classifier.data_processing.loader import load_splits

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Cargar modelo y datos de test

In [ ]:
model = joblib.load("../data/models/artifacts/iris_classifier/model.pkl")
X_train, X_test, y_train, y_test = load_splits()

TARGET_NAMES = ["setosa", "versicolor", "virginica"]
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

## 2. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=TARGET_NAMES))

## 3. Matriz de Confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absoluta
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=TARGET_NAMES, ax=axes[0], cmap="Blues"
)
axes[0].set_title("Matriz de Confusión (absoluta)")

# Normalizada
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=TARGET_NAMES, ax=axes[1],
    cmap="Blues", normalize="true", values_format=".2f"
)
axes[1].set_title("Matriz de Confusión (normalizada)")

plt.tight_layout()
plt.show()

## 4. Curvas ROC (One-vs-Rest)

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(TARGET_NAMES):
    RocCurveDisplay.from_predictions(
        y_test_bin[:, i], y_proba[:, i], name=name, ax=ax
    )
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_title("Curvas ROC — One-vs-Rest")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Curvas Precision-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(TARGET_NAMES):
    PrecisionRecallDisplay.from_predictions(
        y_test_bin[:, i], y_proba[:, i], name=name, ax=ax
    )
ax.set_title("Curvas Precision-Recall — One-vs-Rest")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Análisis de errores

In [ ]:
errors = X_test.copy()
errors["y_true"] = y_test.map(dict(enumerate(TARGET_NAMES))).values
errors["y_pred"] = pd.Series(y_pred).map(dict(enumerate(TARGET_NAMES))).values
errors["correct"] = errors["y_true"] == errors["y_pred"]

misclassified = errors[~errors["correct"]]
print(f"Errores: {len(misclassified)} de {len(errors)} ({len(misclassified)/len(errors)*100:.1f}%)")
misclassified

## 7. Distribución de probabilidades predichas

In [ ]:
max_proba = y_proba.max(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_proba[errors["correct"]], bins=20, alpha=0.7, label="Correctas", color="green")
ax.hist(max_proba[~errors["correct"]], bins=20, alpha=0.7, label="Errores", color="red")
ax.set_xlabel("Probabilidad máxima predicha")
ax.set_ylabel("Frecuencia")
ax.set_title("Confianza del modelo en sus predicciones")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Completar tras ejecutar el notebook con el modelo entrenado.
- Revisar si los errores se concentran en la frontera versicolor/virginica.
- Evaluar si la confianza del modelo es alta en las predicciones correctas y baja en los errores.